In [1]:
import pandas as pd
import geopandas as gpd

# Load cleaned crime data
crime = pd.read_csv('../data/processed/crime_clean.csv')

# Load neighborhood boundaries
neighborhoods = gpd.read_file('../data/raw/neighborhoods.geojson')

print(crime.shape)
print(neighborhoods.shape)
neighborhoods.head()

(236447, 6)
(26, 9)


,OBJECTID,name,acres,neighborhood_id,sqmiles,Shape_Length,Shape_Area,shape_wkt,geometry
0,1,Roslindale,1605.568237,15,2.51,0.169120,0.000709,None,"MULTIPOLYGON (((-71.12593 42.272, -71.12611 42..."
1,2,Jamaica Plain,2519.245394,11,3.94,0.179578,0.001113,None,"POLYGON ((-71.10499 42.32609, -71.10503 42.326..."
2,3,Mission Hill,350.853564,13,0.55,0.059235,0.000155,None,"POLYGON ((-71.09043 42.33576, -71.0905 42.3357..."
3,4,Longwood,188.611947,28,0.29,0.038816,0.000083,None,"POLYGON ((-71.09811 42.33672, -71.09832 42.337..."
4,5,Bay Village,26.539839,33,0.04,0.015625,0.000012,None,"POLYGON ((-71.06663 42.34877, -71.06663 42.348..."


In [9]:
crime_geo = gpd.GeoDataFrame( crime, geometry=gpd.points_from_xy(crime['Long'], crime['Lat']))

In [10]:
print(type(crime_geo))
print(crime_geo.shape)
crime_geo.head()

<class 'geopandas.geodataframe.GeoDataFrame'>
(236447, 7)


,OFFENSE_DESCRIPTION,DISTRICT,SHOOTING,OCCURRED_ON_DATE,Lat,Long,geometry
0,INVESTIGATE PERSON,B3,0,2023-01-27 22:44:00+00,42.271661,-71.099534,POINT (-71.09953 42.27166)
1,VERBAL DISPUTE,B2,0,2023-01-17 20:21:00+00,42.312597,-71.092875,POINT (-71.09288 42.3126)
2,INVESTIGATE PERSON,A1,0,2023-01-24 00:00:00+00,42.365700,-71.052892,POINT (-71.05289 42.3657)
3,INVESTIGATE PROPERTY,B3,0,2023-03-31 17:14:00+00,42.292788,-71.088519,POINT (-71.08852 42.29279)
4,ASSAULT - AGGRAVATED,B2,0,2023-01-26 09:00:00+00,42.310269,-71.089310,POINT (-71.08931 42.31027)


In [11]:
print(crime_geo.crs)
print(neighborhoods.crs)

None
EPSG:4326


In [12]:
crime_geo = crime_geo.set_crs(epsg=4326)
print(crime_geo.crs)

EPSG:4326


In [13]:
crime_with_neighborhoods = gpd.sjoin(
    crime_geo, 
    neighborhoods[['name', 'geometry']], 
    how='left', 
    predicate='within'
)

print(crime_with_neighborhoods.shape)
crime_with_neighborhoods.head()

(236448, 9)


,OFFENSE_DESCRIPTION,DISTRICT,SHOOTING,OCCURRED_ON_DATE,Lat,Long,geometry,index_right,name
0,INVESTIGATE PERSON,B3,0,2023-01-27 22:44:00+00,42.271661,-71.099534,POINT (-71.09953 42.27166),20.0,Mattapan
1,VERBAL DISPUTE,B2,0,2023-01-17 20:21:00+00,42.312597,-71.092875,POINT (-71.09288 42.3126),8.0,Roxbury
2,INVESTIGATE PERSON,A1,0,2023-01-24 00:00:00+00,42.365700,-71.052892,POINT (-71.05289 42.3657),7.0,North End
3,INVESTIGATE PROPERTY,B3,0,2023-03-31 17:14:00+00,42.292788,-71.088519,POINT (-71.08852 42.29279),21.0,Dorchester
4,ASSAULT - AGGRAVATED,B2,0,2023-01-26 09:00:00+00,42.310269,-71.089310,POINT (-71.08931 42.31027),8.0,Roxbury


In [14]:
print(neighborhoods.total_bounds)
print(crime_geo.total_bounds)

[-71.1912491   42.22791113 -70.92277988  42.3969775 ]
[-71.22812796  42.17145601 -70.83688105  42.45998696]


In [15]:
crime_with_neighborhoods = gpd.sjoin(
    crime_geo,
    neighborhoods[['name', 'geometry']],
    how='left',
    predicate='within'
)

print(crime_with_neighborhoods.shape)
crime_with_neighborhoods['name'].value_counts()



(236448, 9)


name
Dorchester                 54640
Roxbury                    28153
South End                  15430
South Boston               14064
Downtown                   13603
Jamaica Plain              13448
East Boston                12496
Brighton                   11415
Hyde Park                  10070
Back Bay                    9813
Mattapan                    8193
West Roxbury                7823
Fenway                      5658
Allston                     5407
Roslindale                  4955
West End                    4103
Charlestown                 3948
Mission Hill                3348
Beacon Hill                 2604
North End                   1971
South Boston Waterfront     1904
Chinatown                   1513
Longwood                    1117
Bay Village                  356
Leather District             313
Harbor Islands                11
Name: count, dtype: int64

In [16]:
crime_neighborhoods = crime_with_neighborhoods[['OFFENSE_DESCRIPTION', 'SHOOTING', 'OCCURRED_ON_DATE', 'Lat', 'Long', 'name']].copy()
crime_neighborhoods = crime_neighborhoods.rename(columns={'name': 'neighborhood'})
crime_neighborhoods.head()

,OFFENSE_DESCRIPTION,SHOOTING,OCCURRED_ON_DATE,Lat,Long,neighborhood
0,INVESTIGATE PERSON,0,2023-01-27 22:44:00+00,42.271661,-71.099534,Mattapan
1,VERBAL DISPUTE,0,2023-01-17 20:21:00+00,42.312597,-71.092875,Roxbury
2,INVESTIGATE PERSON,0,2023-01-24 00:00:00+00,42.365700,-71.052892,North End
3,INVESTIGATE PROPERTY,0,2023-03-31 17:14:00+00,42.292788,-71.088519,Dorchester
4,ASSAULT - AGGRAVATED,0,2023-01-26 09:00:00+00,42.310269,-71.089310,Roxbury


In [17]:
crime_neighborhoods.to_csv('../data/processed/crime_with_neighborhoods.csv', index=False)